# Fast Chess — label positions, then trainThis notebook does the **expensive** half of the data pipeline: running a boundedStockfish search over pre-sampled positions. Sampling already happened locally,because it is cheap; only labelling is worth sending to a bigger machine.### Read this before picking an accelerator**The GPUs cannot speed up labelling.** The teacher is Stockfish, an alpha-betasearch that runs entirely on the CPU. Labelling throughput scales with **CPU coresand nothing else**, so a T4 x2 session labels no faster than a CPU-only session —and CPU-only sessions get the longer time limit, which is what actually matters fora multi-hour job. `kernel-metadata.json` therefore ships with `enable_gpu: false`.Training afterwards *is* GPU work, but this network is tiny: it trains in secondseither way (measured locally: 2.9 s on CPU, 1.8 s on a GTX 1650 Ti). That is not areason to give up the longer CPU session.If you want maximum labelling throughput, the thing to compare is **cores persession**, not the accelerator. Check the resource gauge in the sidebar; whicheveroption reports the most CPUs is the fastest one here. The cell below prints whatthis session actually got, and the labeller sizes itself to that automatically.

In [ ]:
import multiprocessing, os, platform, shutil, subprocess, sys, time
CORES = os.cpu_count() or 2
print('python    ', platform.python_version())
print('cpu cores ', CORES)
try:
    with open('/proc/meminfo') as fh:
        print('memory    ', int(fh.readline().split()[1]) / 1048576, 'GiB')
except OSError:
    pass
# nvidia-smi is absent entirely on a CPU-only session, so check before calling it.
gpu = ''
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
print('gpu       ', gpu or 'none (fine: labelling is CPU-only)')

## 1. Dependencies and Stockfish

Installs python-chess if the image lacks it. Stockfish is tried in order:
an already-installed binary, the distro package, then the official static
build. The last needs internet enabled for the notebook.


In [ ]:
# python-chess is not in the Kaggle image. On PyPI the package is named `chess`.
try:
    import chess
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chess'], check=True)
    import chess
print('python-chess', chess.__version__)

# Debian installs the binary into /usr/games, which is not on PATH in every image,
# so look there explicitly rather than trusting `which` alone.
os.environ['PATH'] = os.environ.get('PATH', '') + ':/usr/games:/usr/local/games'
CANDIDATES = ('/usr/games/stockfish', '/usr/local/bin/stockfish', '/usr/bin/stockfish')

def find_stockfish():
    return shutil.which('stockfish') or next((p for p in CANDIDATES if os.path.exists(p)), None)

STOCKFISH = find_stockfish()
if not STOCKFISH:
    subprocess.run('apt-get -qq update && DEBIAN_FRONTEND=noninteractive '
                   'apt-get -qq install -y stockfish', shell=True, check=False)
    STOCKFISH = find_stockfish()
if not STOCKFISH:
    # Last resort: the official static build. Locate the binary after extracting
    # rather than assuming the archive's internal layout.
    import glob, tarfile
    url = ('https://github.com/official-stockfish/Stockfish/releases/latest/download/'
           'stockfish-ubuntu-x86-64-avx2.tar')
    subprocess.run(f'curl -fsSL {url} -o /tmp/sf.tar', shell=True, check=True)
    if not tarfile.is_tarfile('/tmp/sf.tar'):
        raise SystemExit('Download was not a tar archive; check internet access '
                         'and that the release asset name is still current.')
    tarfile.open('/tmp/sf.tar').extractall('/tmp/sf')
    binaries = [p for p in glob.glob('/tmp/sf/**/stockfish*', recursive=True)
                if os.path.isfile(p) and not p.endswith(('.nnue', '.txt', '.md'))]
    if not binaries:
        raise SystemExit('No stockfish binary inside the archive.')
    STOCKFISH = binaries[0]
    os.chmod(STOCKFISH, 0o755)
print('stockfish:', STOCKFISH)
print(subprocess.run([STOCKFISH], input='uci\nquit\n', capture_output=True, text=True)
      .stdout.splitlines()[0])

## 2. Code and positionsBoth come from the input dataset built by `kaggle/package.sh`, so the notebook needsno repository checkout and stays in step with the local code.

In [ ]:
import glob, gzip, shutil as sh
from pathlib import Path

WORK = Path('/kaggle/working')
# Recursive: tolerates the bundle being nested (e.g. uploading the whole
# kaggle/ folder rather than just kaggle/upload/).
matches = glob.glob('/kaggle/input/**/fastchess/features.py', recursive=True)
if not matches:
    raise SystemExit('Attach the fast-chess-positions dataset (Add Input) before running.')
source = Path(matches[0]).parent
sh.rmtree(WORK / 'fastchess', ignore_errors=True)
sh.copytree(source, WORK / 'fastchess')
sys.path.insert(0, str(WORK))

packed = next(iter(sorted(glob.glob(str(source.parent / 'positions.tsv*'))
                          or glob.glob('/kaggle/input/**/positions.tsv*', recursive=True))), None)
if packed is None:
    raise SystemExit('positions.tsv(.gz) missing from the dataset.')
target = WORK / 'positions.tsv'
if not target.exists():
    opener = gzip.open if packed.endswith('.gz') else open
    with opener(packed, 'rt') as src, open(target, 'w') as dst:
        sh.copyfileobj(src, dst)
rows = sum(1 for line in open(target) if not line.startswith('#'))
print(f'{rows:,} positions ready at {target}')

## 3. Label`--workers` is set to the cores this session actually has. Shards are written to`/kaggle/working/shards`, so if the session dies you can **re-run this cell and itresumes** instead of starting over — worth knowing on a multi-hour job.`NODES` is the per-position Stockfish budget and is the main quality/time dial.Higher means better labels and proportionally more time. Start by leaving `LIMIT`small to measure the real rate on this machine, then set it to 0 for the full run.

In [ ]:
NODES = 10000      # Stockfish nodes per position
LIMIT = 2000       # 0 = label everything; a small value first gives a timing sample
WORKERS = CORES

from fastchess import label
started = time.time()
label.main(['--positions', str(WORK / 'positions.tsv'),
            '--out', str(WORK / 'teacher.npz'),
            '--nodes', str(NODES), '--workers', str(WORKERS),
            '--limit', str(LIMIT),
            '--shards', str(WORK / 'shards'),
            '--stockfish', STOCKFISH])
labelled = sum(sum(1 for _ in open(shard)) for shard in (WORK / 'shards').glob('shard-*.txt'))
rate = (LIMIT or rows) / max(time.time() - started, 1e-3)
print(f'\n{labelled:,} labelled in total; this run averaged ~{rate:.0f} positions/s '
      f'on {WORKERS} workers')
print(f'at that rate the full {rows:,} positions take ~{rows / rate / 3600:.1f} h '
      '(a resumed run finishes faster, since completed positions are skipped)')

If that projection fits comfortably inside the session limit, set `LIMIT = 0` aboveand re-run the cell — already-labelled positions are skipped. If it does not fit,either lower `NODES`, or label a prefix now and raise `LIMIT` in a later session:the shards persist in the notebook output and resume cleanly.

## 4. TrainOnly useful as a sanity check that the labels train sensibly — the real training isseconds long and you will likely re-run it locally anyway. `--device auto` uses aGPU if this session happens to have one.

In [ ]:
import shutil as sh

# Re-running after relabelling must not resume a checkpoint fitted to the old
# labels, so clear any previous run first.
sh.rmtree(WORK / 'runs' / 'teacher', ignore_errors=True)

from fastchess import train
train.main(['--data', str(WORK / 'teacher.npz'),
            '--out', str(WORK / 'runs' / 'teacher'),
            '--epochs', '60', '--device', 'auto',
            '--threads', str(max(1, WORKERS // 2)), '--minutes', '20'])

## 5. Package for downloadShards are deleted once the `.npz` exists, so the notebook output stays small. The`.npz` is the artefact you actually want.

In [ ]:
import json
import numpy as np

sh.rmtree(WORK / 'shards', ignore_errors=True)
sh.rmtree(WORK / 'fastchess', ignore_errors=True)
(WORK / 'positions.tsv').unlink(missing_ok=True)

with np.load(WORK / 'teacher.npz', allow_pickle=False) as data:
    meta = json.loads(str(data['metadata']))
    print(f"positions {data['X'].shape[0]:,}  games {len(np.unique(data['game'])):,}  "
          f"cp range {int(data['cp'].min())}..{int(data['cp'].max())}")
    print(f"nodes/position {meta['nodes']:,}  labelling {meta['seconds'] / 3600:.2f} h")
for path in sorted(WORK.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(WORK)!s:<40} {path.stat().st_size / 1048576:8.1f} MiB')

## 6. Back on your machine```bashkaggle kernels output <username>/fast-chess-label-and-train -p data/bash run.sh train --data data/teacher.npz --out runs/teacher500k --epochs 60bash run.sh benchmark --games 10 --parallel 4 --out runs/teacher500k-match```Compare against the current model before trusting it — validation loss is notplaying strength:```bashbash run.sh evaluate --opponent material --games 12 --seconds 1 --parallel 4 \  --model runs/teacher500k/best.npz --out runs/teacher500k-vs-material.json```